In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.feature_selection import SelectKBest, f_classif
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [3]:
class TitanicML:
    #初始化
    def __init__(self):
        self.train_data=None
        self.test_data=None
        self.load_data()
    #加载数据
    def load_data(self):
        self.train_data = pd.read_csv('datasets/titanic_train.csv')
        self.test_data = pd.read_csv('datasets/titanic_test.csv')
        print(f'训练集大小:{self.train_data.shape}')
        print(f'测试集大小:{self.test_data.shape}')
    #预处理数据
    def preprocess_data(self,data):
        data=data.copy()
        #缺失值处理
        data['Age']=data['Age'].fillna(data['Age'].median())
        data['Fare']=data['Fare'].fillna(data['Fare'].median())
        data['Embarked']=data['Embarked'].fillna('S')
        data['Cabin']=data['Cabin'].fillna('U')
        #非数值特征替换
        data['Sex']=data['Sex'].map({'male':0,'female':1})
        data['Embarked']=data['Embarked'].map({'S':0,'C':1,'Q':2})
        #添加新特征
        data['FamilySize']=data['SibSp']+data['Parch']+1
        data['IsAlone']=(data['FamilySize']==1).astype(int)
        #提取名字中的称谓
        data['Title']=data['Name'].str.extract('([A-Za-z]+)\.',expand=False)
        data['Title']=data['Title'].replace(['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],'Rare')
        data['Title']=data['Title'].replace('Mlle','Miss')
        data['Title']=data['Title'].replace('Ms', 'Miss')
        data['Title']=data['Title'].replace('Mme','Mrs')
        title_mapping={'Mr':1,'Miss':2,'Mrs':3,'Master':4,'Rare':5}
        data['Title']=data['Title'].map(title_mapping)
        data['Title']=data['Title'].fillna(0)
        #返回处理好的数据
        return data

    def get_features(self,data):
        #选取特征
        features=['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked','FamilySize','IsAlone','Title']
        return data[features]

    def train_svm(self):
        #传入训练集处理数据
        train_processed=self.preprocess_data(self.train_data)
        #将目标标签分离,X为所选特征的训练数据
        X=self.get_features(train_processed)
        y=train_processed['Survived']
        #SVC模型参数设置
        alg=SVC(C=2,kernel='rbf',gamma=10,decision_function_shape='ovr')
        #K折交叉验证评估参数设置
        kf=KFold(n_splits=3,shuffle=True,random_state=42)
        #所有得分保存
        test_scores=[]
        #3折交叉验证
        for train_index, test_index in kf.split(X):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            #拟合训练集的3折后的训练集
            alg.fit(X_train,y_train)
            #预测训练集3折后的测试集得到y_pred
            y_pred=alg.predict(X_test)
            #将分数保存到test_scores列表
            test_scores.append(accuracy_score(y_test,y_pred))
        #取平均分数
        avg_test_score=np.mean(test_scores)
        print(f'SVM训练完成，平均准确率：{avg_test_score:.4f}')
        #返回平均分数
        return avg_test_score
    
    #KNN模型评估，平均得分
    def train_knn(self):
        train_processed=self.preprocess_data(self.train_data)
        X=self.get_features(train_processed)
        y=train_processed['Survived']
        alg=KNeighborsClassifier(n_neighbors=10,)
        kf=KFold(n_splits=3,shuffle=True,random_state=42)
        test_scores=[]
        for train_index, test_index in kf.split(X):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            alg.fit(X_train,y_train)
            y_pred=alg.predict(X_test)
            test_scores.append(accuracy_score(y_test,y_pred))

        avg_test_score=np.mean(test_scores)
        print(f'KNN模型训练完成，平均准确率:{avg_test_score:.4f}')
        return avg_test_score

    #特征重要性及可视化
    def feature_importance(self):
        train_processed=self.preprocess_data(self.train_data)
        X=self.get_features(train_processed)
        y=train_processed['Survived']
        selector=SelectKBest(f_classif,k='all')
        selector.fit(X,y)

        #特征得分与特征名获取
        score=-np.log10(selector.pvalues_)
        features=X.columns

        print('\n特征重要性排名:')
        for feature,score in zip(features,score):
            print(f'   {feature}:{score:.4f}')

        #特征重要性可视化，图片保存
        plt.figure(figsize=(10,6))
        plt.bar(range(len(features)),score)
        plt.xticks(range(len(features)),features)
        plt.xlabel('Feature')
        plt.ylabel('Score')
        plt.title('Feature Importance Analysis')
        plt.tight_layout()
        plt.savefig('feature_importance.png')
        print('特征重要性图表已保存为feature_importance.png')

        return dict(zip(features,score))


    #运行所有模型，得到分数
    def run_all_models(self):
        #svm模型的得分
        svm_score=self.train_svm()
        print(f'SVM:   {svm_score:.4f}')
        #knn模型的得分
        knn_score=self.train_knn()
        print(f'KNN:   {knn_score:.4f}')

if __name__ == '__main__':
    titanic_ml=TitanicML()
    titanic_ml.run_all_models()



训练集大小:(891, 12)
测试集大小:(418, 11)
SVM训练完成，平均准确率：0.6263
SVM:   0.6263
KNN模型训练完成，平均准确率:0.7003
KNN:   0.7003
